In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")


True

In [ ]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationHistory

lazy_frame = pl.scan_parquet("data/raw/alarm_history_dump.parquet")

SimpleTimeCorrelationHistory.train(lazy_frame)

In [ ]:
from utils.node_summary import node_summary

summary = node_summary(os.getenv("HISTORY_DB_PATH"))

summary_sorted = summary.sort("num_subgraphs", descending=True)

In [1]:
summary_formated = (
    summary_sorted
    .with_columns(
        pl.when((pl.col("num_subgraphs") == 0) & (pl.col("num_nodes") > 0))
        .then(1)
        .otherwise(pl.col("num_subgraphs"))
        .alias("num_subgraphs")
    )
    .with_columns(
        (pl.col("num_nodes") / pl.col("num_subgraphs"))
        .fill_nan(1.0) # Garante o tratamento caso exista algum caso 0 / 0
        .alias("nodes_por_subgraph")
    )
    .with_columns(
        pl.when(pl.col("num_nodes") <= 1)
        .then(0.0) # Se só tem 1 alarme, não há conexões possíveis, logo a densidade é 0%
        .otherwise(
            (2.0 * pl.col("num_edges")) / (pl.col("num_nodes") * (pl.col("num_nodes") - 1))
        )
        # Trava de segurança para garantir que o valor fique estritamente entre 0.0 e 1.0
        .clip(0.0, 1.0)
        .alias("density")
    )
    .rename({
        "node_id": "ID do Node",
        "num_nodes": "Total de Alarmes",
        "num_edges": "Total de Correlações",
        "num_subgraphs": "Total de Incidentes",
        "nodes_por_subgraph": "Média de Nós por Incidente",
        "density": "Densidade (Density)"
    })
)

media_nos_por_incidente_geral = summary_formated["Média de alarmes por Incidente"].mean()
media_incidente_geral = summary_formated["Total de Incidentes"].mean()
media_density_geral = summary_formated["Densidade (Density)"].mean()

print(f"➔ Média geral de alarmes por incidente: {media_nos_por_incidente_geral:.2f}")
print(f"➔ Média geral de incidentes por Node: {media_incidente_geral:.2f}")
print(f"➔ Densidade média dos Nodes: {media_density_geral * 100:.1f}%\n")

summary_formated

with pl.Config(tbl_rows=-1):
    display(summary_formated)

NameError: name 'summary_sorted' is not defined